# 🎨 Stable Diffusion Image Generator

[![GitHub](https://img.shields.io/badge/GitHub-ThanhTrunggDEV%2FStableDiffusion-blue)](https://github.com/ThanhTrunggDEV/StableDiffusion)

Generate images from text using Stable Diffusion with Flask on Google Colab!

---

## 📋 Instructions

1. **Get ngrok token**: Sign up at [ngrok.com](https://ngrok.com/) and get your authtoken
2. **Run all cells** in order (Runtime > Run all)
3. **Click the ngrok URL** that appears to access the web interface
4. **Generate images** using the web UI!

⚠️ **Note**: First run will download Stable Diffusion model (~4GB), this takes time!

## 🔧 Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repository
import os

repo_url = "https://github.com/ThanhTrunggDEV/StableDiffusion.git"
project_dir = "/content/StableDiffusion"

if os.path.exists(project_dir):
    print("📁 Repository already exists, pulling latest changes...")
    !cd {project_dir} && git pull
else:
    print("📥 Cloning repository...")
    !git clone {repo_url} {project_dir}

# Change to project directory
%cd {project_dir}
!ls -la

## 📦 Step 2: Install Dependencies

In [ ]:
# Install Python dependencies
print("📦 Installing dependencies... This may take 5-10 minutes.")
!pip install -q flask diffusers transformers torch torchvision accelerate safetensors pillow python-dotenv pyngrok
print("✅ Dependencies installed successfully!")

## 🌐 Step 3: Setup ngrok

Get your authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok, conf
import getpass

# Set your ngrok authtoken
print("🔑 Enter your ngrok authtoken:")
print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("ngrok authtoken: ")

# Configure ngrok
conf.get_default().auth_token = ngrok_token
print("✅ ngrok configured!")

## 🚀 Step 4: Start the Application

In [ ]:
import threading
import time
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

# Create app_colab.py with modifications for Colab
with open('app_colab.py', 'w') as f:
    f.write('''
from flask import Flask, render_template, request, jsonify, send_from_directory
from config import Config
from utils.image_generator import ImageGenerator
import os
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

app = Flask(__name__)
app.config.from_object(Config)
Config.init_app()

generator = None

def get_generator():
    global generator
    if generator is None:
        generator = ImageGenerator(
            model_id=Config.MODEL_ID,
            device="cuda"  # Force CUDA for Colab
        )
    return generator

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/gallery')
def gallery():
    generated_dir = Config.GENERATED_DIR
    images = []
    
    if generated_dir.exists():
        for file in generated_dir.glob('*.png'):
            images.append({
                'filename': file.name,
                'url': f'/static/generated/{file.name}',
                'created': file.stat().st_mtime
            })
        images.sort(key=lambda x: x['created'], reverse=True)
    
    return render_template('gallery.html', images=images)

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.get_json()
        prompt = data.get('prompt', '').strip()
        negative_prompt = data.get('negative_prompt', '').strip()
        width = int(data.get('width', Config.DEFAULT_WIDTH))
        height = int(data.get('height', Config.DEFAULT_HEIGHT))
        steps = int(data.get('steps', Config.DEFAULT_STEPS))
        guidance_scale = float(data.get('guidance_scale', Config.DEFAULT_GUIDANCE_SCALE))
        seed = data.get('seed')
        
        if not prompt:
            return jsonify({'error': 'Prompt is required'}), 400
        
        if width > Config.MAX_WIDTH or height > Config.MAX_HEIGHT:
            return jsonify({'error': f'Image dimensions too large'}), 400
        
        if steps > Config.MAX_STEPS:
            return jsonify({'error': f'Too many steps'}), 400
        
        if seed:
            try:
                seed = int(seed)
            except ValueError:
                seed = None
        
        logger.info(f"Generating image: {prompt}")
        
        gen = get_generator()
        image = gen.generate_image(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=width,
            height=height,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            seed=seed
        )
        
        filename = gen.save_image(image, Config.GENERATED_DIR)
        
        return jsonify({
            'success': True,
            'image_url': f'/static/generated/{filename}',
            'filename': filename
        })
        
    except Exception as e:
        logger.error(f"Error generating image: {e}")
        return jsonify({'error': str(e)}), 500

@app.route('/api/settings')
def get_settings():
    return jsonify({
        'default_width': Config.DEFAULT_WIDTH,
        'default_height': Config.DEFAULT_HEIGHT,
        'default_steps': Config.DEFAULT_STEPS,
        'default_guidance_scale': Config.DEFAULT_GUIDANCE_SCALE,
        'max_width': Config.MAX_WIDTH,
        'max_height': Config.MAX_HEIGHT,
        'max_steps': Config.MAX_STEPS,
        'model_id': Config.MODEL_ID,
        'device': "cuda"
    })

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
''')

print("🚀 Starting Flask application...")
print("⏳ Please wait, this may take a few minutes...")

# Function to run Flask app
def run_app():
    !python app_colab.py

# Start Flask in a separate thread
thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

# Wait for Flask to start
time.sleep(5)

# Start ngrok tunnel
public_url = ngrok.connect(5000, bind_tls=True)

print("\n" + "="*80)
print("🎉 APPLICATION IS READY!")
print("="*80)
print(f"\n🌐 Public URL: {public_url}")
print("\n📝 How to use:")
print("   1. Click the URL above to open the web interface")
print("   2. Enter a text prompt (e.g., 'A beautiful sunset over mountains')")
print("   3. Click 'Generate Image' and wait ~10-30 seconds")
print("   4. Your image will appear on the right side!")
print("\n⚠️  Note: First generation will be slower due to model loading")
print("="*80)
print("\n💡 Keep this notebook running to maintain the server!")
print("\n🔗 Repository: https://github.com/ThanhTrunggDEV/StableDiffusion")

## 💡 Tips for Better Results

### Good Prompt Examples:
```
A majestic castle on a cliff, sunset, fantasy art, highly detailed, 8k
Portrait of a cyberpunk girl, neon lights, futuristic city, digital art
Beautiful landscape with mountains and lake, morning mist, photorealistic
Cute cartoon cat playing with yarn, kawaii style, colorful
```

### Negative Prompt Suggestions:
```
blurry, low quality, distorted, deformed, ugly, bad anatomy, watermark
```

### Parameter Recommendations:
- **Steps**: 30-50 for good quality (higher = slower but better)
- **Guidance Scale**: 7-12 (how closely to follow prompt)
- **Size**: 512x512 is fastest, 768x768 for more detail

## 🛠️ Management Commands

In [ ]:
# View recent logs (optional)
!tail -n 50 /content/StableDiffusion/*.log 2>/dev/null || echo "No logs yet"

In [ ]:
# Check generated images
!ls -lh /content/StableDiffusion/static/generated/ 2>/dev/null || echo "No images generated yet"

In [ ]:
# Monitor GPU usage (run this cell periodically)
!nvidia-smi

In [ ]:
# Restart ngrok tunnel (if needed)
from pyngrok import ngrok

ngrok.kill()
print("🔄 Restarting ngrok...")
time.sleep(2)
public_url = ngrok.connect(5000, bind_tls=True)
print(f"\n🌐 New Public URL: {public_url}")

## 🔧 Troubleshooting

### Issue: "CUDA out of memory"
**Solution**: Reduce image size to 384x384 or 256x256

### Issue: "Connection refused"
**Solution**: Re-run the "Start Application" cell

### Issue: "ngrok tunnel closed"
**Solution**: Re-run the "Restart ngrok" cell above

### Issue: Generation is very slow
**Solution**: 
- Check you're using GPU (Runtime > Change runtime type > GPU)
- Reduce steps to 25-30
- Reduce image size

---

### 🌟 Enjoy Creating Art with AI!

Made with ❤️ by [ThanhTrunggDEV](https://github.com/ThanhTrunggDEV)

⭐ Star the repo if you find this useful: [GitHub Repository](https://github.com/ThanhTrunggDEV/StableDiffusion)